# CTRL-MATH AIMO3 — Final Submission Notebook

**Competition**: AI Mathematical Olympiad Progress Prize 3 (AIMO3)

**Pipeline**:
1. Install offline dependencies
2. Load all CTRL-MATH modules
3. Initialize vLLM/Transformers with Qwen2.5-Math-14B + LoRA
4. Read test problems
5. Solve each problem via unified solver
6. Generate submission.csv

**Target**: >40/50 on AIMO3 public LB

In [ ]:
# ── Cell 0: Offline pip installs + env setup ────────────────────────────────
import subprocess, sys, os

# Kaggle offline mode: install from pre-downloaded wheels
WHEELS_DIR = "/kaggle/input/ctrlmath-wheels"
if os.path.isdir(WHEELS_DIR):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-index",
        "--find-links", WHEELS_DIR, "-q",
        "numpy", "numba", "sympy", "torch", "transformers",
        "accelerate", "bitsandbytes", "peft", "trl", "z3-solver",
        "sentence-transformers", "scipy",
    ])
    print("✅ Offline packages installed from wheels.")
else:
    print("⚠️ No offline wheels found. Using pre-installed packages.")

# Set Numba cache directory
os.environ["NUMBA_CACHE_DIR"] = "/kaggle/working/.numba_cache"
os.makedirs("/kaggle/working/.numba_cache", exist_ok=True)
print("✅ Environment configured.")

In [ ]:
# ── Cell 1: Load all CTRL-MATH modules ──────────────────────────────────────
import time
t0 = time.time()

# Core number theory (Numba JIT — triggers warm-up)
from cell_02a_numba_nt import (
    vp_jit, vp_factorial_jit, vp_binomial_jit,
    dirichlet_conv_safe, powmod_batch, modinv_batch,
    fib_jit, sigma_k_sieve, poly_mul_ntt, ntt,
)

# GPU acceleration
from cell_02b_cupy_gpu import GPUArithmetic

# Problem parsing
from cell_03_mog_parser import MOGParser

# Domain solvers
from cell_04_transform_engine import TransformEngine, TransformResult
from cell_04b_linear_recurrence import LinearRecurrenceSolver
from cell_04c_combinatorics import CombinatoricsSolver
from cell_04d_number_theory import NumberTheorySolver
from cell_04e_gf_solver import GFSolver
from cell_04f_geometry import GeometrySolver
from cell_04g_geometry_prover import geometry_tool, GeometryTool

# MCTS + MPC
from cell_06_mcts import MCTSEngine
from cell_06_mpc_planner import MPCPlanner, solve_aimo3

# LLM executor
from cell_07_llm_executor_v5 import LLMExecutorV5

# PRM + Kalman
from cell_08_prm import ProcessRewardModel
from cell_08_kalman import KalmanBeliefState

# Verification
from cell_09_mathrag import MathRAG
from cell_09b_z3_parallel import ParallelZ3Checker
from cell_10_template_store import TemplateStore
from cell_11_answer_extractor import AnswerExtractor

# Orchestration
from cell_12_time_allocator import TimeAllocator
from cell_13_self_consistency import SelfConsistencyChecker
from cell_14_verification_ladder import VerificationLadder
from cell_15_orchestrator_v5 import SolveOrchestrator

print(f"✅ All CTRL-MATH modules loaded in {time.time() - t0:.1f}s")

In [ ]:
# ── Cell 2: Initialize LLM (Qwen2.5-Math-14B + LoRA) ────────────────────────
import torch

# Model paths (Kaggle datasets)
PRIMARY_MODEL = "/kaggle/input/qwen2.5-math-14b-instruct"
ENSEMBLE_MODEL = "/kaggle/input/deepseek-math-7b-instruct"
LORA_ADAPTER = "/kaggle/input/ctrlmath-aimo3-lora"

# Fallback to HuggingFace Hub if not in Kaggle
import os
if not os.path.isdir(PRIMARY_MODEL):
    PRIMARY_MODEL = "Qwen/Qwen2.5-Math-14B-Instruct"
if not os.path.isdir(ENSEMBLE_MODEL):
    ENSEMBLE_MODEL = "deepseek-ai/DeepSeek-Math-7B-Instruct"
if not os.path.isdir(LORA_ADAPTER):
    LORA_ADAPTER = None

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("⚠️ No GPU available — will use CPU-only mode")

# Initialize LLM executor
try:
    llm = LLMExecutorV5(
        primary_model=PRIMARY_MODEL,
        ensemble_model=ENSEMBLE_MODEL,
        lora_adapter=LORA_ADAPTER,
        load_in_4bit=True,
        use_flash_attn=True,
    )
except Exception as e:
    print(f"⚠️ LLM loading failed: {e}")
    print("Using CPU-only mode (no LLM inference)")
    llm = None

# Initialize PRM
prm = ProcessRewardModel()

# Initialize Z3 checker
z3_checker = ParallelZ3Checker(n_workers=4)

print("✅ Models initialized.")

In [ ]:
# ── Cell 3: Build orchestrator + read test data ──────────────────────────────
import pandas as pd

# Build the full solver orchestrator
orchestrator = SolveOrchestrator(
    llm=llm,
    prm=prm,
    z3_checker=z3_checker,
    total_seconds=9 * 3600.0,  # 9-hour competition limit
    n_problems=50,
    mcts_sims=128,
)

# Read AIMO3 test data
TEST_PATHS = [
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv",
    "/kaggle/input/aimo-2025-progress-prize-3/test.csv",
]

test_df = None
for path in TEST_PATHS:
    if os.path.exists(path):
        test_df = pd.read_csv(path)
        print(f"✅ Loaded {len(test_df)} test problems from {path}")
        break

if test_df is None:
    # Fallback: create a dummy test dataframe for local testing
    print("⚠️ No test data found. Creating dummy test set for validation.")
    test_df = pd.DataFrame({
        "id": [0, 1, 2],
        "problem": [
            "Find the remainder when 2^100 is divided by 7.",
            "How many ways can you choose 3 items from 10?",
            "What is the sum of the first 100 positive integers?",
        ],
    })

print(f"Test problems: {len(test_df)}")
print(test_df.head())

In [ ]:
# ── Cell 4: Solve all problems ───────────────────────────────────────────────
import asyncio

predictions = []
total_time = 0.0

for idx, row in test_df.iterrows():
    problem_id = str(row.get("id", idx))
    problem_text = str(row.get("problem", ""))
    
    t0 = time.time()
    
    # Try the full orchestrator first
    try:
        answer = orchestrator.solve_problem(problem_id, problem_text)
    except Exception as e:
        print(f"  ⚠️ Orchestrator failed for #{problem_id}: {e}")
        answer = 0
    
    elapsed = time.time() - t0
    total_time += elapsed
    
    # Ensure answer is a non-negative integer
    try:
        answer = max(0, int(answer))
    except (TypeError, ValueError):
        answer = 0
    
    predictions.append(answer)
    
    if (idx + 1) % 5 == 0 or idx == len(test_df) - 1:
        avg_time = total_time / (idx + 1)
        print(f"  [{idx+1}/{len(test_df)}] Problem {problem_id}: answer={answer} "
              f"({elapsed:.1f}s, avg={avg_time:.1f}s)")

print(f"\n✅ All {len(predictions)} problems solved in {total_time:.1f}s "
      f"(avg {total_time/max(len(predictions),1):.1f}s/problem)")

In [ ]:
# ── Cell 5: Generate submission.csv ──────────────────────────────────────────
submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": predictions,
})

# Validate: all answers must be non-negative integers in [0, 999]
submission["answer"] = submission["answer"].astype(int).clip(lower=0, upper=999)

submission.to_csv("submission.csv", index=False)
print(f"✅ submission.csv saved with {len(submission)} predictions.")
print(submission.head(10))

# Summary statistics
print(f"\nAnswer statistics:")
print(f"  Non-zero answers: {(submission['answer'] > 0).sum()}/{len(submission)}")
print(f"  Unique answers:   {submission['answer'].nunique()}")
print(f"  Answer range:     [{submission['answer'].min()}, {submission['answer'].max()}]")